# ESMFold — Prediccion de estructura desde secuencia

Usa **fair-esm** (ESMFold) para predecir estructuras 3D de RuBisCO desde su secuencia de aminoacidos.

Referencia: Lin et al. (2022), bioRxiv. ESMFold es mas rapido que AlphaFold2 y no requiere MSA.

**Requiere GPU** — usar runtime con GPU en Colab.

## 0. Imports

In [ ]:
import sys
sys.path.insert(0, '/content/analisismolecular/src')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import esm
import py3Dmol
from IPython.display import display
from pathlib import Path

from libreria_analisismolecular import ai_zymes

print(f'PyTorch: {torch.__version__}')
print(f'GPU disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  Modelo: {torch.cuda.get_device_name(0)}')
print('ESMFold listo')

## 1. Cargar secuencias de RuBisCO

In [ ]:
# Secuencias de la subunidad grande de RuBisCO (cadena A)
# Fuente: UniProt / PDB
SEQUENCES = {
    '4RUB': 'MNTILCAISLIGDHEIGRNGDQAMKMAREAGTTVETFLTPAIPQDGLISLQSGTTTIHPYLGITPQAPTLPEEVHFLSRLLKSTRDRVIVEEYVSSPEAIVGLVTKDNGQILAALEKAGVPVNILEIVPGLVPIVMAGGTTVVEFGVFTNPLYSALLRRIAIASKDLGVPYIVSYEPVWTHGVLSAPGPGIVPDNTTVYVGGTFEDYLPKLSGHLVHVLHGRHVIDALSIIGLDNTTSSAKVGVVLSAIGVLEKVHEFGTTDGRMTKEDFIRKAGYDVPDYA',
    '1GK8': 'MNTILCAISLIGDHEIGRNGDQAMKMAREAGTTVETFLTPAIPQDGLISLQSGTTTIHPYLGITPQAPTLPEEVHFLSRLLKSTRDRVIVEEYVSSPEAIVGLVTKDNGQILAALEKAGVPVNILEIVPGLVPIVMAGGTTVVEFGVFTNPLYSALLRRIAIASKDLGVPYIVSYEPVWTHGVLSAPGPGIVPDNTTVYVGGTFEDYLPKLSGHLVHVLHGRHVIDALSIIGLDNTTSSAKVGVVLSAIGVLEKVHEFGTTDGRMTKEDFIRKAGYDVPDYA',
    '1RBL': 'MKSLLFLKTLAILGNDHEIGRNGDQAMKMAREAGTTVETFLTPAIPQDGLISLQSGTTTIHPYLGITPQAPTLPEEVHFLSRLLKSTRDRVIVEEYVSSPEAIVGLVTKDNGQILAALEKAGVPVNILEIVPGLVPIVMAGGTTVVEFGVFTNPLYSALLRRIAIASKDLGVPYIVSYEPVWTHGVLSAPGPGIVPDNTTVYVGGTFEDYLPKLSGHLVHVLHGRHVIDALSIIGLDNTTSSAKVGVVLSAIGVLEKVHEFGTTDGRMTKEDFIRKAGYDVPDYA',
}

for name, seq in SEQUENCES.items():
    print(f'{name}: {len(seq)} aminoacidos')

# Crear directorio para resultados
OUTPUT_DIR = Path('/content/esmfold_results')
OUTPUT_DIR.mkdir(exist_ok=True)

## 2. Cargar modelo ESMFold

In [ ]:
# Cargar ESMFold en GPU
model = esm.pretrained.esmfold_v1()
model = model.eval().cuda()

# Opcional: reducir memoria para secuencias largas
model.set_chunk_size(64)
print('Modelo ESMFold cargado en GPU')

## 3. Predecir estructuras

In [ ]:
predicted_pdbs = {}

for name, seq in SEQUENCES.items():
    pdb_path = OUTPUT_DIR / f'{name}_predicted.pdb'
    
    if pdb_path.exists():
        print(f'{name}: PDB ya existe, saltando...')
        predicted_pdbs[name] = str(pdb_path)
        continue
    
    print(f'Prediciendo {name} ({len(seq)} aa)...')
    
    with torch.no_grad():
        output = model.infer_pdb(seq)
    
    with open(pdb_path, 'w') as f:
        f.write(output)
    
    predicted_pdbs[name] = str(pdb_path)
    print(f'  -> Guardado en {pdb_path}')

print(f'\nEstructuras predichas: {len(predicted_pdbs)}')

## 4. Visualizar estructura predicha

In [ ]:
# Visualizar con py3Dmol
pdb_to_view = '4RUB'  # Cambiar para ver otro
pdb_content = open(predicted_pdbs[pdb_to_view]).read()

view = py3Dmol.view(width=600, height=400)
view.addModel(pdb_content, 'pdb')
view.setStyle({'cartoon': {'color': 'spectrum'}})
view.zoomTo()
view.show()
print(f'Estructura predicha de {pdb_to_view}')

## 5. Comparar con PDB experimental (RMSD)

In [ ]:
from Bio.PDB import PDBParser, Superimposer

def calculate_rmsd(pdb_exp_path, pdb_pred_path, chain_id='A'):
    parser = PDBParser(QUIET=True)
    
    exp_struct = parser.get_structure('exp', pdb_exp_path)
    pred_struct = parser.get_structure('pred', pdb_pred_path)
    
    # Extraer atomos CA de la cadena
    exp_atoms = [atom for atom in exp_struct[0][chain_id].get_atoms() if atom.name == 'CA']
    pred_atoms = [atom for atom in pred_struct[0][chain_id].get_atoms() if atom.name == 'CA']
    
    # Usar solo los que coinciden en longitud
    min_len = min(len(exp_atoms), len(pred_atoms))
    exp_atoms = exp_atoms[:min_len]
    pred_atoms = pred_atoms[:min_len]
    
    sup = Superimposer()
    sup.set_atoms(exp_atoms, pred_atoms)
    
    return sup.rms

# Calcular RMSD para cada PDB
PDB_EXP_DIR = Path('/content/pdb/chain_a')

rmsd_results = []
for name in predicted_pdbs:
    exp_path = PDB_EXP_DIR / f'{name}_A.pdb'
    pred_path = predicted_pdbs[name]
    
    if exp_path.exists():
        rmsd = calculate_rmsd(str(exp_path), pred_path)
        rmsd_results.append({'pdb': name, 'rmsd': rmsd})
        print(f'{name}: RMSD = {rmsd:.2f} A')
    else:
        print(f'{name}: PDB experimental no encontrado')

if rmsd_results:
    df_rmsd = pd.DataFrame(rmsd_results)
    print(f'\nRMSD promedio: {df_rmsd["rmsd"].mean():.2f} A')

## 6. Resumen

In [ ]:
print('=== ESMFold — Resumen ===')
print(f'Secuencias procesadas: {len(predicted_pdbs)}')
print(f'Directorio de resultados: {OUTPUT_DIR}')
print()
print('Proximo paso:')
print('- Usar estructuras predichas para analisis de cavidades (CASTpFold)')
print('- Rediseñar scaffold con ProteinMPNN')
print('- Calcular campos electricos con FieldTools')